# MoodFit V2-2 - Load V2 checkpoint and fine-tune more layers
기존 V2 체크포인트를 불러온 뒤, 마지막 3개 ConvNeXt 모듈을 추가로 미세조정합니다.

In [ ]:
import copy
import json
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Sampler, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")
device = torch.device("cuda:0")
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
print(f"사용 디바이스: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
dataset_path = "CNN_train_test/CNN_train_test_image"
target_per_class = 800
batch_size = 32
workers = 4
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05), interpolation=transforms.InterpolationMode.BILINEAR, fill=(128, 128, 128)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
full_dataset = datasets.ImageFolder(dataset_path, transform=eval_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
targets = np.asarray(full_dataset.targets)
train_ids, val_ids = train_test_split(np.arange(len(full_dataset)), test_size=0.2, stratify=targets, random_state=SEED)
train_source = copy.copy(full_dataset)
train_source.transform = train_transforms
train_dataset = Subset(train_source, train_ids.tolist())
val_dataset = Subset(full_dataset, val_ids.tolist())
class BalancedClassSampler(Sampler):
    def __init__(self, labels, samples_per_class, seed=42):
        labels = np.asarray(labels)
        self.class_positions = [np.flatnonzero(labels == label) for label in np.unique(labels)]
        self.samples_per_class = int(samples_per_class)
        self.seed = int(seed)
        self.epoch = 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.class_positions) * self.samples_per_class
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        positions = []
        for candidates in self.class_positions:
            chosen = rng.permutation(candidates)[:self.samples_per_class]
            if len(chosen) < self.samples_per_class:
                extra = rng.choice(candidates, self.samples_per_class - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
            positions.extend(chosen.tolist())
        rng.shuffle(positions)
        return iter(positions)
train_sampler = BalancedClassSampler(targets[train_ids], target_per_class, SEED)
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
print(f"학습: {num_classes}개 클래스 x {target_per_class}회 = {len(train_sampler):,}회/epoch")
print(f"배치: {len(train_loader):,}개/epoch | 검증: {len(val_loader.dataset):,}장")


In [ ]:
checkpoint_path = Path("Model/fashion_mood_model_pytorch2_epoch25_last3features.pth")
if not checkpoint_path.exists():
    raise FileNotFoundError(f"먼저 기존 V2 모델을 저장하세요: {checkpoint_path}")
weights = models.ConvNeXt_Tiny_Weights.DEFAULT
mood_model = models.convnext_tiny(weights=weights)
for param in mood_model.parameters():
    param.requires_grad = False
in_features = mood_model.classifier[2].in_features
mood_model.classifier = nn.Sequential(nn.Flatten(1), nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
mood_model.load_state_dict(state_dict)
feature_modules = list(mood_model.features.children())
for module in feature_modules[-5:]:
    for param in module.parameters():
        param.requires_grad = True
mood_model = mood_model.to(device)
criterion = nn.CrossEntropyLoss()
backbone_params = [param for param in mood_model.features.parameters() if param.requires_grad]
optimizer = optim.AdamW([{"params": backbone_params, "lr": 1e-6}, {"params": mood_model.classifier.parameters(), "lr": 1e-4}], weight_decay=1e-4)
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
print("기존 V2 체크포인트 로드 완료")
print("마지막 3개 ConvNeXt 모듈을 추가 학습합니다.")


In [ ]:
additional_epochs = 20
base_epoch = 25
train_loss_history, val_loss_history = [], []
train_acc_history, val_acc_history = [], []
val_macro_f1_history, val_balanced_acc_history = [], []
model_dir = Path("Model")
model_dir.mkdir(exist_ok=True)
log_path = model_dir / "v2-2_more_layers_training_log.txt"
best_path = model_dir / "fashion_mood_model_pytorch2-2_more_layers_best.pth"
last_path = model_dir / "fashion_mood_model_pytorch2-2_more_layers_last.pth"
metadata_path = model_dir / "fashion_mood_model_pytorch2-2_more_layers.json"
with log_path.open("a", encoding="utf-8") as log_file:
    log_file.write("\n\n" + "=" * 100 + "\n")
    log_file.write(f"V2-2: loaded epoch {base_epoch}, additional epochs {additional_epochs}\n")
    log_file.write("=" * 100 + "\n")
best_macro_f1 = -1.0
best_epoch = base_epoch
for extra_epoch in range(additional_epochs):
    actual_epoch = base_epoch + extra_epoch + 1
    train_sampler.set_epoch(actual_epoch)
    mood_model.train()
    for module in feature_modules[:-3]:
        module.eval()
    for module in feature_modules[-5:]:
        module.train()
    mood_model.classifier.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {actual_epoch} [Train]", unit="batch"):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            outputs = mood_model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        total_train += labels.size(0)
        correct_train += (outputs.argmax(1) == labels).sum().item()
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    mood_model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    actual_labels, predicted_labels = [], []
    with torch.inference_mode():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {actual_epoch} [Valid]", unit="batch"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                outputs = mood_model(inputs)
                loss = criterion(outputs, labels)
            predicted = outputs.argmax(1)
            val_running_loss += loss.item() * labels.size(0)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(predicted.cpu().tolist())
    val_loss = val_running_loss / total_val
    val_acc = correct_val / total_val
    val_macro_f1 = f1_score(actual_labels, predicted_labels, labels=list(range(num_classes)), average="macro", zero_division=0)
    val_balanced_acc = balanced_accuracy_score(actual_labels, predicted_labels)
    log_line = f"Epoch [{actual_epoch}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%} | Val Macro F1: {val_macro_f1:.4f} | Val Balanced Acc: {val_balanced_acc:.2%}"
    print(log_line, flush=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(log_line + "\n")
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)
    val_macro_f1_history.append(val_macro_f1)
    val_balanced_acc_history.append(val_balanced_acc)
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = actual_epoch
        torch.save(mood_model.state_dict(), best_path)
        print(f"최고 모델 저장: {best_path}")
torch.save(mood_model.state_dict(), last_path)
metadata_path.write_text(json.dumps({"checkpoint": str(checkpoint_path), "base_epoch": base_epoch, "additional_epochs": additional_epochs, "unfrozen_feature_modules": 5, "best_epoch": best_epoch, "best_val_macro_f1": best_macro_f1, "class_names": class_names}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"V2-2 완료: best epoch={best_epoch}, best Macro F1={best_macro_f1:.4f}")
